
# GIS Property Intelligence — Address-Driven Spatial RAG

**Supported input formats:** FileGDB · Shapefile · CSV · GeoJSON · JSON

**Architecture:**
```
Input File (any format)
        ↓
DataLoader (auto-detect)
        ↓
Normalise → AddressPoint
        ↓
Address → Geocode → Lat/Lon
        ↓
Spatial Search (BallTree)
        ↓
Top-K Nearby Features
        ↓
Build Context
        ↓
Embedding Retrieval (ChromaDB)
        ↓
Gemma4
        ↓
Answer
```

## Cell 1 — Install Dependencies

In [1]:
%pip install \
    ollama \
    chromadb \
    pydantic \
    requests \
    pandas \
    numpy \
    geopandas \
    shapely \
    scikit-learn \
    pyproj \
    fiona

## Cell 2 — Configuration

In [ ]:
import json
import math
import os
import pathlib
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import chromadb
from pydantic import BaseModel, Field, field_validator
from typing import Any, Optional
from sklearn.neighbors import BallTree

# ── Ollama ────────────────────────────────────────────────────────────────────
OLLAMA_HOST = "http://10.10.10.100:11434"
EMBED_MODEL = "nomic-embed-text-v2-moe:latest"
LLM_MODEL   = "gemma4:latest"

# ── Retrieval ─────────────────────────────────────────────────────────────────
TOP_K        = 10   # BallTree neighbours
CHROMA_TOP_K = 5    # ChromaDB documents

# ── ChromaDB ─────────────────────────────────────────────────────────────────
CHROMA_PATH     = "./chroma_gis"
COLLECTION_NAME = "address_points"

# ── Column name aliases ───────────────────────────────────────────────────────
# Maps common field name variants → canonical name used internally.
# Add your own dataset's field names here if they differ.
COLUMN_ALIASES: dict[str, list[str]] = {
    "address":   ["address", "addr", "full_address", "site_address", "siteaddress",
                  "street_address", "ADDRESS", "ADDR", "FULL_ADDRESS"],
    "city":      ["city", "municipality", "CITY", "MUNICIPALITY", "muni"],
    "state":     ["state", "st", "STATE", "ST", "state_abbr"],
    "zipcode":   ["zipcode", "zip", "zip_code", "postal_code", "ZIPCODE",
                  "ZIP", "ZIP_CODE", "POSTAL_CODE"],
    "latitude":  ["latitude", "lat", "y", "LAT", "LATITUDE", "Y", "POINT_Y"],
    "longitude": ["longitude", "lon", "lng", "x", "LON", "LONGITUDE",
                  "LNG", "X", "POINT_X"],
}

print("Configuration loaded.")

## Cell 3 — Universal Data Loader

Supports **FileGDB**, **Shapefile**, **CSV**, **GeoJSON**, and **JSON**.
Pass any one of these paths to `load_dataset()` and it returns a
normalised `list[dict]` ready for Pydantic validation.

In [ ]:
def _resolve_alias(columns: list[str], field: str) -> str | None:
    """Return the first column name that matches a known alias for `field`."""
    for alias in COLUMN_ALIASES.get(field, [field]):
        if alias in columns:
            return alias
    return None


def _extract_lat_lon_from_geometry(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    If latitude/longitude columns are absent, derive them from the geometry.
    Handles Points directly; for Polygons/Lines the centroid is used.
    Re-projects to WGS-84 (EPSG:4326) first if the CRS is different.
    """
    if gdf.crs and gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(epsg=4326)

    geom = gdf.geometry
    # Use centroid for any geometry type (centroid of a Point is itself)
    centroids = geom.centroid
    gdf = gdf.copy()
    gdf["latitude"]  = centroids.y
    gdf["longitude"] = centroids.x
    return gdf


def _gdf_to_records(gdf: gpd.GeoDataFrame) -> list[dict]:
    """
    Convert a GeoDataFrame to plain dicts, extracting lat/lon from geometry
    when explicit coordinate columns are absent, and moving all non-canonical
    fields into an 'attributes' sub-dict.
    """
    cols = list(gdf.columns)

    lat_col = _resolve_alias(cols, "latitude")
    lon_col = _resolve_alias(cols, "longitude")

    # Derive from geometry if coordinate columns are missing
    if (lat_col is None or lon_col is None) and gdf.geometry is not None:
        gdf = _extract_lat_lon_from_geometry(gdf)
        lat_col, lon_col = "latitude", "longitude"

    # Drop geometry column before iterating — it isn't JSON-serialisable
    gdf_plain = gdf.drop(columns=[gdf.geometry.name], errors="ignore")
    flat_cols = list(gdf_plain.columns)

    canonical = {f: _resolve_alias(flat_cols, f)
                 for f in ("address", "city", "state", "zipcode", "latitude", "longitude")}

    records = []
    for _, row in gdf_plain.iterrows():
        rec: dict[str, Any] = {}
        attrs: dict[str, Any] = {}

        for field, src_col in canonical.items():
            rec[field] = row[src_col] if src_col and src_col in row else None

        # Everything else → attributes
        for col in flat_cols:
            if col not in canonical.values():
                val = row[col]
                # Convert numpy scalars to plain Python types
                if hasattr(val, "item"):
                    val = val.item()
                attrs[col] = val
        rec["attributes"] = attrs
        records.append(rec)
    return records


def _load_json(path: str) -> list[dict]:
    """
    Load a plain JSON file.  Handles three shapes:
      - list of objects                → used directly
      - {'features': [...]}            → GeoJSON-like, delegated to geopandas
      - single object with nested list → tries common wrapper keys
    """
    with open(path) as f:
        data = json.load(f)

    if isinstance(data, list):
        return data

    if isinstance(data, dict):
        # GeoJSON FeatureCollection → hand off to geopandas
        if data.get("type") == "FeatureCollection" or "features" in data:
            gdf = gpd.read_file(path)
            return _gdf_to_records(gdf)

        # Common wrapper keys: data, records, results, items, rows
        for wrapper_key in ("data", "records", "results", "items", "rows", "features"):
            if wrapper_key in data and isinstance(data[wrapper_key], list):
                return data[wrapper_key]

        # Single object — wrap in a list
        return [data]

    raise ValueError(f"Unsupported JSON root type: {type(data)}")


def _load_csv(path: str) -> list[dict]:
    df = pd.read_csv(path, low_memory=False)
    # Try to detect and parse a geometry / WKT column
    wkt_col = next((c for c in df.columns if c.lower() in ("geometry", "wkt", "shape", "geom")), None)
    if wkt_col:
        from shapely import wkt as shapely_wkt
        try:
            df["_geom"] = df[wkt_col].apply(
                lambda v: shapely_wkt.loads(v) if pd.notna(v) else None
            )
            gdf = gpd.GeoDataFrame(df, geometry="_geom", crs="EPSG:4326")
            return _gdf_to_records(gdf)
        except Exception:
            pass  # Fall through to plain dict conversion

    cols = list(df.columns)
    canonical = {f: _resolve_alias(cols, f)
                 for f in ("address", "city", "state", "zipcode", "latitude", "longitude")}

    records = []
    for _, row in df.iterrows():
        rec: dict[str, Any] = {}
        attrs: dict[str, Any] = {}
        for field, src_col in canonical.items():
            rec[field] = row[src_col] if src_col and src_col in row else None
        for col in cols:
            if col not in canonical.values():
                val = row[col]
                if hasattr(val, "item"):
                    val = val.item()
                attrs[col] = val
        rec["attributes"] = attrs
        records.append(rec)
    return records


def load_dataset(path: str, layer: str | None = None) -> list[dict]:
    """
    Universal entry point.  Auto-detects the format from the file extension
    (or directory structure for FileGDB) and returns a normalised list[dict].

    Parameters
    ----------
    path  : str  Path to the dataset file or directory.
    layer : str  Optional layer name for multi-layer formats (FileGDB, GPKG).
                 If None, the first layer is used.

    Supported formats
    -----------------
    .gdb             → Esri FileGDB   (via Fiona / GeoPandas)
    .shp             → Shapefile      (via GeoPandas)
    .csv             → CSV            (via Pandas + optional WKT geometry)
    .geojson / .json → GeoJSON / JSON (via GeoPandas / json)
    """
    p    = pathlib.Path(path)
    ext  = p.suffix.lower()

    # ── FileGDB (.gdb directory) ───────────────────────────────────────────────
    if ext == ".gdb" or (p.is_dir() and path.endswith(".gdb")):
        import fiona
        available_layers = fiona.listlayers(path)
        chosen = layer if layer and layer in available_layers else available_layers[0]
        print(f"[FileGDB] layers: {available_layers}")
        print(f"[FileGDB] loading layer: '{chosen}'")
        gdf = gpd.read_file(path, layer=chosen)
        records = _gdf_to_records(gdf)

    # ── Shapefile (.shp) ──────────────────────────────────────────────────────
    elif ext == ".shp":
        print(f"[Shapefile] loading: {p.name}")
        gdf = gpd.read_file(path)
        records = _gdf_to_records(gdf)

    # ── GeoJSON (.geojson) ────────────────────────────────────────────────────
    elif ext == ".geojson":
        print(f"[GeoJSON] loading: {p.name}")
        gdf = gpd.read_file(path)
        records = _gdf_to_records(gdf)

    # ── JSON (.json) ──────────────────────────────────────────────────────────
    elif ext == ".json":
        print(f"[JSON] loading: {p.name}")
        records = _load_json(path)

    # ── CSV (.csv) ────────────────────────────────────────────────────────────
    elif ext == ".csv":
        print(f"[CSV] loading: {p.name}")
        records = _load_csv(path)

    else:
        raise ValueError(
            f"Unsupported format '{ext}'.  "
            "Supported: .gdb, .shp, .geojson, .json, .csv"
        )

    print(f"Loaded {len(records):,} raw records from '{p.name}'.")
    return records


# ─────────────────────────────────────────────────────────────────────────────
# USAGE — point to any one of your datasets:
# ─────────────────────────────────────────────────────────────────────────────
# raw_records = load_dataset("addresspoints.gdb", layer="AddressPoints")
# raw_records = load_dataset("addresspoints.shp")
# raw_records = load_dataset("addresspoints.csv")
# raw_records = load_dataset("addresspoints.geojson")
# raw_records = load_dataset("addresspoints.json")

DATA_FILE = "Address_Points.geojson"   # ← change this to your file
raw_records = load_dataset(DATA_FILE)
print(f"\nSample raw record:")
print(json.dumps(raw_records[0], indent=2, default=str))

## Cell 4 — Pydantic Model & Validation

In [ ]:
class AddressPoint(BaseModel):
    """Validated, normalised representation of one address-point record."""

    address:    str
    city:       str
    state:      str
    zipcode:    str
    latitude:   float
    longitude:  float
    attributes: dict[str, Any] = Field(default_factory=dict)

    # ── Coerce None / missing strings to empty string ─────────────────────────
    @field_validator("address", "city", "state", "zipcode", mode="before")
    @classmethod
    def coerce_str(cls, v: Any) -> str:
        return str(v) if v is not None else ""

    # ── Coerce NaN / None floats to 0.0 ──────────────────────────────────────
    @field_validator("latitude", "longitude", mode="before")
    @classmethod
    def coerce_float(cls, v: Any) -> float:
        try:
            f = float(v)
            return f if math.isfinite(f) else 0.0
        except (TypeError, ValueError):
            return 0.0

    @property
    def full_address(self) -> str:
        return f"{self.address}, {self.city}, {self.state} {self.zipcode}"

    @property
    def coords_rad(self) -> tuple[float, float]:
        """(lat_rad, lon_rad) — required by BallTree haversine metric."""
        return (math.radians(self.latitude), math.radians(self.longitude))


def validate_records(records: list[dict]) -> tuple[list[AddressPoint], list[dict]]:
    """
    Validate and normalise raw records.
    Returns (valid_points, failed_records).
    Records with lat=0 and lon=0 are skipped (likely missing geometry).
    """
    valid, failed = [], []
    for i, rec in enumerate(records):
        try:
            point = AddressPoint(**rec)
            if point.latitude == 0.0 and point.longitude == 0.0:
                failed.append({"index": i, "reason": "zero lat/lon", "record": rec})
            else:
                valid.append(point)
        except Exception as exc:
            failed.append({"index": i, "reason": str(exc), "record": rec})
    return valid, failed


validated, failed = validate_records(raw_records)

print(f"Validated : {len(validated):,} records")
print(f"Failed    : {len(failed):,} records")
if failed:
    print("Sample failures:")
    for f in failed[:3]:
        print(f"  [{f['index']}] {f['reason']}")

## Cell 5 — Build Spatial Index (BallTree)

In [ ]:
EARTH_RADIUS_MILES = 3_958.8

# Coordinate matrix in radians (haversine requires radians)
coords = np.array(
    [[math.radians(p.latitude), math.radians(p.longitude)] for p in validated],
    dtype=np.float64,
)

tree = BallTree(coords, metric="haversine")


def spatial_query(lat: float, lon: float, k: int = TOP_K) -> list[dict]:
    """
    Return the k nearest validated address points to (lat, lon),
    each enriched with a 'distance_miles' key.
    """
    query = np.array([[math.radians(lat), math.radians(lon)]])
    distances, indices = tree.query(query, k=min(k, len(validated)))

    results = []
    for dist_rad, idx in zip(distances[0], indices[0]):
        rec = validated[idx].model_dump()
        rec["distance_miles"] = round(dist_rad * EARTH_RADIUS_MILES, 4)
        results.append(rec)
    return results


print(f"BallTree spatial index built.")
print(f"  Records indexed : {len(coords):,}")
print(f"  Default top-k   : {TOP_K}")

## Cell 6 — Geocoder & Geo-Document Builder

In [ ]:
def geocode(address: str) -> tuple[float, float] | None:
    """
    Resolve a free-text address to (lat, lon) via Nominatim.
    Returns None when the address cannot be resolved.
    """
    url     = "https://nominatim.openstreetmap.org/search"
    params  = {"q": address, "format": "json", "limit": 1}
    headers = {"User-Agent": "GIS-Property-Intelligence/1.0"}
    try:
        resp = requests.get(url, params=params, headers=headers, timeout=10)
        resp.raise_for_status()
        hits = resp.json()
        if hits:
            return float(hits[0]["lat"]), float(hits[0]["lon"])
    except Exception as exc:
        print(f"[geocode] error: {exc}")
    return None


def build_geo_document(point: dict, rank: int) -> str:
    """
    Convert one address-point record into a human-readable text document
    for embedding.  Raw coordinates and geometry blobs are never embedded.
    """
    attrs = point.get("attributes", {})

    doc = (
        f"Address      : {point.get('address','')}, {point.get('city','')}, "
        f"{point.get('state','')} {point.get('zipcode','')}\n"
        f"Rank         : {rank}\n"
        f"Distance     : {point.get('distance_miles', 'N/A')} miles\n"
        f"Zoning       : {attrs.get('zoning', 'Unknown')}\n"
        f"Flood Zone   : {attrs.get('flood_zone', 'Unknown')}\n"
        f"County       : {attrs.get('county', 'Unknown')}\n"
        f"Land Use     : {attrs.get('land_use', 'Unknown')}\n"
        f"Nearest Hwy  : {attrs.get('nearest_highway', 'Unknown')}\n"
        f"Hwy Distance : {attrs.get('highway_distance_miles', 'Unknown')} miles\n"
        f"School Dist  : {attrs.get('school_district', 'Unknown')}\n"
        f"Parcel ID    : {attrs.get('parcel_id', 'Unknown')}\n"
    )

    # Append extra attributes not already rendered above
    known = {
        "zoning", "flood_zone", "county", "land_use",
        "nearest_highway", "highway_distance_miles",
        "school_district", "parcel_id",
    }
    extra = {k: v for k, v in attrs.items() if k not in known}
    if extra:
        doc += "Extra        : " + ", ".join(f"{k}={v}" for k, v in extra.items()) + "\n"

    return doc.strip()


print("Geocoder and geo-document builder ready.")

## Cell 7 — Build ChromaDB Collection

In [ ]:
def embed(texts: list[str]) -> list[list[float]]:
    """Batch-embed strings using nomic-embed-text-v2-moe via Ollama."""
    url  = f"{OLLAMA_HOST}/api/embed"
    body = {"model": EMBED_MODEL, "input": texts}
    resp = requests.post(url, json=body, timeout=120)
    resp.raise_for_status()
    return resp.json()["embeddings"]


# ── ChromaDB ──────────────────────────────────────────────────────────────────
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
try:
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass
collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

# ── Index all validated address points ───────────────────────────────────────
BATCH_SIZE = 64
for batch_start in range(0, len(validated), BATCH_SIZE):
    batch   = validated[batch_start : batch_start + BATCH_SIZE]
    docs    = [build_geo_document(p.model_dump(), i) for i, p in enumerate(batch)]
    ids     = [f"addr_{batch_start + i}" for i in range(len(batch))]
    metas   = [
        {
            "address"  : p.address,
            "city"     : p.city,
            "state"    : p.state,
            "zipcode"  : p.zipcode,
            "latitude" : p.latitude,
            "longitude": p.longitude,
        }
        for p in batch
    ]
    vectors = embed(docs)
    collection.add(ids=ids, documents=docs, embeddings=vectors, metadatas=metas)

    if (batch_start // BATCH_SIZE) % 10 == 0:
        done = min(batch_start + BATCH_SIZE, len(validated))
        print(f"  Indexed {done:,} / {len(validated):,}")

print(f"\nChromaDB collection '{COLLECTION_NAME}' — {collection.count():,} documents.")

## Cell 8 — Main Query Function: `ask_address`

In [ ]:
def build_spatial_context(lat: float, lon: float, nearby: list[dict]) -> dict:
    """Aggregate top-k neighbours into a structured context dict for the LLM."""
    highways, zones, schools = [], set(), set()

    for rec in nearby:
        attrs = rec.get("attributes", {})
        if attrs.get("nearest_highway"):
            highways.append({
                "name"          : attrs["nearest_highway"],
                "distance_miles": attrs.get("highway_distance_miles", "N/A"),
            })
        if attrs.get("zoning"):
            zones.add(attrs["zoning"])
        if attrs.get("school_district"):
            schools.add(attrs["school_district"])

    # De-duplicate highways by name; keep the closest occurrence
    seen, deduped = set(), []
    for h in sorted(
        highways,
        key=lambda x: float(x["distance_miles"])
                      if str(x["distance_miles"]).replace(".", "").isdigit()
                      else 9999,
    ):
        if h["name"] not in seen:
            deduped.append(h)
            seen.add(h["name"])

    closest       = nearby[0] if nearby else {}
    closest_attrs = closest.get("attributes", {})

    return {
        "coordinates"            : {"lat": lat, "lon": lon},
        "nearest_address"        : closest.get("address", "N/A"),
        "nearest_distance_miles" : closest.get("distance_miles", "N/A"),
        "zoning"                 : closest_attrs.get("zoning", "Unknown"),
        "flood_zone"             : closest_attrs.get("flood_zone", "Unknown"),
        "county"                 : closest_attrs.get("county", "Unknown"),
        "land_use"               : closest_attrs.get("land_use", "Unknown"),
        "school_districts"       : sorted(schools),
        "nearest_highways"       : deduped[:5],
        "surrounding_zones"      : sorted(zones),
    }


def vector_retrieval(question: str, k: int = CHROMA_TOP_K) -> list[str]:
    """Embed the question and return the k most relevant geo-documents."""
    q_vec   = embed([question])[0]
    results = collection.query(query_embeddings=[q_vec], n_results=k)
    return results["documents"][0] if results["documents"] else []


def ask_address(address: str, question: str, verbose: bool = False) -> str:
    """
    Full pipeline:
      address → geocode → lat/lon
             → spatial top-k → context
      question → vector retrieval
             → prompt → Gemma4 → answer
    """
    # Step 1 — Geocode
    coords = geocode(address)
    if coords is None:
        return f"[ERROR] Could not geocode: '{address}'"
    lat, lon = coords
    if verbose:
        print(f"[geocode]  {address} → ({lat:.6f}, {lon:.6f})")

    # Step 2 — Spatial top-k
    nearby = spatial_query(lat, lon, k=TOP_K)
    if verbose:
        print(f"[spatial]  {len(nearby)} neighbours, closest {nearby[0]['distance_miles']} mi")

    # Step 3 — Structured spatial context
    context = build_spatial_context(lat, lon, nearby)

    # Step 4 — Vector retrieval
    retrieved = vector_retrieval(question)
    if verbose:
        print(f"[chroma]   {len(retrieved)} documents retrieved")
    retrieved_text = "\n\n---\n\n".join(retrieved) if retrieved else "No documents retrieved."

    # Step 5 — Build prompt
    prompt = f"""You are a GIS Property Intelligence Assistant.

Address: {address}
Geocoded Coordinates: ({lat:.6f}, {lon:.6f})

Question: {question}

=== Spatial Context (nearest neighbours) ===
{json.dumps(context, indent=2)}

=== Retrieved Documents ===
{retrieved_text}

Instructions:
- Answer the question using ONLY the supplied context and documents.
- Be specific — cite distances, zone codes, and flood categories where available.
- If information is unavailable in the context, state that clearly.
- Do not invent or infer data not present in the context.
"""

    if verbose:
        print("[gemma4]   sending prompt…")

    # Step 6 — Gemma4
    url  = f"{OLLAMA_HOST}/api/generate"
    body = {"model": LLM_MODEL, "prompt": prompt, "stream": False}
    resp = requests.post(url, json=body, timeout=120)
    resp.raise_for_status()
    return resp.json()["response"].strip()


print("ask_address() is ready.")

## Usage Examples

In [ ]:
# Example 1 — Transportation corridors
answer = ask_address(
    address  = "949 Sapphire St",
    question = "What transportation corridors are nearby?",
    verbose  = True,
)
print(answer)

In [ ]:
# Example 2 — Commercial zoning
answer = ask_address(
    address  = "949 Sapphire St",
    question = "Can I build a commercial property here?",
)
print(answer)

In [ ]:
# Example 3 — Flood risk
answer = ask_address(
    address  = "949 Sapphire St",
    question = "What flood risks affect this property?",
)
print(answer)

In [ ]:
# Example 4 — School district
answer = ask_address(
    address  = "949 Sapphire St",
    question = "Which school district serves this address?",
)
print(answer)